# FIR Transportes — Data Lake em Python (PySpark / Databricks)

In [ ]:
# ===== CONFIGURAÇÃO =====
CATALOG = "workspace"   
SCHEMA  = "fir_lake"
VOLUME  = "bronze_raw"
ARQUIVO = "ibge_populacao_municipios.csv"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

def T(nome):                   # nome totalmente qualificado de tabela
    return f"{CATALOG}.{SCHEMA}.{nome}"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print("ok:", CATALOG, SCHEMA, VOLUME)

## 1. Bronze — dados brutos, append-only
Equivale ao `raw_populacao_municipio` do SQL: tudo como texto + metadados de ingestão

In [ ]:
import os

CSV_TEXTO = """codigo_municipio,municipio,uf,populacao,ano_referencia,fonte
3304557,Rio de Janeiro,RJ,6211223,2022,IBGE - Censo Demografico 2022
3303302,Niterói,RJ,481749,2022,IBGE - Censo Demografico 2022
3304904,São Gonçalo,RJ,896744,2022,IBGE - Censo Demografico 2022
3301702,Duque de Caxias,RJ,808161,2022,IBGE - Censo Demografico 2022
3303500,Nova Iguaçu,RJ,785867,2022,IBGE - Censo Demografico 2022
"""

def gravar(nome, texto):
    """Grava o arquivo no Volume só se ainda não existir (idempotente, como no laboratório)."""
    caminho = f"{VOLUME_PATH}/{nome}"
    if os.path.exists(caminho):
        print("já existe:", caminho)
    else:
        with open(caminho, "w", encoding="utf-8") as f:
            f.write(texto)
        print("gravado:", caminho)
    return caminho

# Na prática: baixe a tabela oficial do IBGE (SIDRA 4714 - Censo 2022) e faça upload no Volume.
# Aqui gravamos o mesmo CSV usado na versão SQL.
caminho_csv = gravar(ARQUIVO, CSV_TEXTO)

In [ ]:
from pyspark.sql import functions as F

def ingerir_bronze_ibge(arquivo):
    """Lê o CSV (tudo string) e faz append na bronze, ignorando arquivos já ingeridos."""
    df = (spark.read.option("header", True).option("encoding", "UTF-8")
          .csv(f"{VOLUME_PATH}/{arquivo}")          # sem inferSchema: tudo string
          .withColumn("_arquivo", F.lit(arquivo))
          .withColumn("_ingestao", F.current_timestamp()))
    tabela = T("bronze_ibge_populacao")
    if spark.catalog.tableExists(tabela):
        ja = [r[0] for r in spark.table(tabela).select("_arquivo").distinct().collect()]
        if arquivo in ja:
            print("arquivo já ingerido, nada a fazer:", arquivo)
            return
        df.write.mode("append").saveAsTable(tabela)
    else:
        df.write.mode("overwrite").saveAsTable(tabela)
    print("ingerido:", arquivo, "-", df.count(), "linhas")

ingerir_bronze_ibge(ARQUIVO)
display(spark.table(T("bronze_ibge_populacao")))

### Extrato do OLTP (`oper_fir.Endereco`)
No Databricks não existe o PostgreSQL da FIR. Em um cenário real você exportaria as tabelas do OLTP para CSV e faria upload no Volume (mesmo padrão acima). Para manter o notebook autocontido, os 15 endereços do DML estão embutidos aqui como DataFrame.

In [ ]:
enderecos = [
 (1,'20031170','Av. Rio Branco','1','Rio de Janeiro','RJ','Centro'),
 (2,'22410003','Av. Vieira Souto','200','Rio de Janeiro','RJ','Ipanema'),
 (3,'24210200','Rua Gavioes','45','Niteroi','RJ','Icarai'),
 (4,'20520054','Rua Conde de Bonfim','350','Rio de Janeiro','RJ','Tijuca'),
 (5,'22640102','Av. das Americas','4666','Rio de Janeiro','RJ','Barra da Tijuca'),
 (6,'21351000','Estrada do Portela','99','Rio de Janeiro','RJ','Madureira'),
 (7,'24020070','Av. Amaral Peixoto','200','Niteroi','RJ','Centro'),
 (8,'24240000','Rua Mem de Sa','150','Niteroi','RJ','Santa Rosa'),
 (9,'24440000','Av. Presidente Kennedy','500','Sao Goncalo','RJ','Centro'),
 (10,'24710000','Rua Coronel Moreira Cesar','120','Sao Goncalo','RJ','Alcantara'),
 (11,'25010020','Av. Presidente Vargas','600','Duque de Caxias','RJ','Centro'),
 (12,'25215000','Rua General Argolo','45','Duque de Caxias','RJ','Jardim Primavera'),
 (13,'26210060','Av. Governador Roberto Silveira','700','Nova Iguacu','RJ','Centro'),
 (14,'26255000','Rua Dr. Mario Guimaraes','80','Nova Iguacu','RJ','Moqueta'),
 (15,'21040360','Av. Brasil','9000','Rio de Janeiro','RJ','Bonsucesso'),
]
cols = ["EnderecoID","EnderecoCEP","EnderecoLogradouro","EnderecoNumero","EnderecoMunicipio","EnderecoUF","EnderecoPontoReferencia"]
(spark.createDataFrame(enderecos, cols)
      .withColumn("_ingestao", F.current_timestamp())
      .write.mode("overwrite").saveAsTable(T("bronze_oper_endereco")))

# Sugestões de rota (EnderecoID de referência, indicação O=origem / D=destino)
sugestoes = [(1,3,'O'),(2,1,'D'),(3,9,'O'),(4,10,'D'),(5,13,'O'),(6,11,'D'),(7,7,'O'),(8,3,'D')]
(spark.createDataFrame(sugestoes, ["SugestaoID","EnderecoID","SugestaoIndicacao"])
      .write.mode("overwrite").saveAsTable(T("bronze_oper_sugestaorota")))
print("bronze OLTP ok")

## 2. Silver — limpeza, tipagem e chave normalizada
Equivale à *materialized view* `curated_populacao_municipio` e à função `chave_municipio()`.

In [ ]:
from pyspark.sql import Window

COM_ACENTO = "áàâãäéèêëíìîïóòôõöúùûüçñÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

def chave_municipio(nome, uf):
    """'Niterói'/'RJ' -> 'NITEROI/RJ' (mesma regra do SQL: sem acento, espaços colapsados, maiúsculas)."""
    n = F.regexp_replace(F.trim(nome), r"\s+", " ")
    n = F.translate(n, COM_ACENTO, SEM_ACENTO)
    return F.concat(F.upper(n), F.lit("/"), F.upper(F.trim(uf)))

def construir_silver_populacao():
    bruto = spark.table(T("bronze_ibge_populacao"))
    # versão mais recente de cada município (equivale ao DISTINCT ON do PostgreSQL)
    w = Window.partitionBy("codigo_municipio").orderBy(F.col("_ingestao").desc(), F.col("ano_referencia").desc())
    silver = (bruto
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
        .select(
            F.col("codigo_municipio").alias("CodigoMunicipioIBGE"),
            F.trim("municipio").alias("NomeMunicipio"),
            F.upper(F.trim("uf")).alias("UF"),
            F.regexp_replace("populacao", r"[^0-9]", "").cast("int").alias("PopulacaoResidente"),
            F.col("ano_referencia").cast("int").alias("AnoReferencia"),
            chave_municipio(F.col("municipio"), F.col("uf")).alias("ChaveMunicipio"),
            F.col("_ingestao").alias("DataIngestao"))
        .filter(F.col("PopulacaoResidente").isNotNull()))
    silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T("silver_populacao_municipio"))

construir_silver_populacao()
display(spark.table(T("silver_populacao_municipio")))

## 3. Gold — dimensão e indicadores
`dim_endereco` replica `dw_fir.Endereco` (com população e porte). A população entra por **LEFT JOIN** na chave normalizada: município sem match fica com NULL / 'Nao informado'.

In [ ]:
def construir_dim_endereco():
    e = spark.table(T("bronze_oper_endereco"))
    p = spark.table(T("silver_populacao_municipio"))
    dim = (e.alias("e")
        .join(p.alias("p"), chave_municipio(F.col("e.EnderecoMunicipio"), F.col("e.EnderecoUF")) == F.col("p.ChaveMunicipio"), "left")
        .select("e.EnderecoID","e.EnderecoCEP","e.EnderecoLogradouro","e.EnderecoNumero",
                "e.EnderecoMunicipio","e.EnderecoUF","e.EnderecoPontoReferencia",
                F.col("p.CodigoMunicipioIBGE").alias("CodigoIBGEMunicipio"),
                F.col("p.PopulacaoResidente").alias("PopulacaoMunicipio"),
                F.col("p.AnoReferencia").alias("AnoPopulacaoMunicipio"))
        .withColumn("PorteMunicipio",
            F.when(F.col("PopulacaoMunicipio").isNull(), "Nao informado")
             .when(F.col("PopulacaoMunicipio") < 100000, "Pequeno (ate 100 mil)")
             .when(F.col("PopulacaoMunicipio") <= 500000, "Medio (100 a 500 mil)")
             .otherwise("Grande (acima de 500 mil)")))
    dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T("dim_endereco"))

construir_dim_endereco()
display(spark.table(T("dim_endereco")).orderBy("EnderecoID"))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("gold_penetracao_municipio")} AS
SELECT d.EnderecoMunicipio              AS Municipio,
       d.EnderecoUF                     AS UF,
       MAX(d.PopulacaoMunicipio)        AS Populacao,      -- MAX: a população se repete por endereço, nunca SUM
       MAX(d.PorteMunicipio)            AS Porte,
       COUNT(s.SugestaoID)              AS Sugestoes,
       ROUND(COUNT(s.SugestaoID) * 100000.0 / NULLIF(MAX(d.PopulacaoMunicipio), 0), 2) AS SugestoesPor100mil
FROM {T("dim_endereco")} d
LEFT JOIN {T("bronze_oper_sugestaorota")} s ON s.EnderecoID = d.EnderecoID
GROUP BY d.EnderecoMunicipio, d.EnderecoUF
""")
display(spark.table(T("gold_penetracao_municipio")).orderBy(F.desc("SugestoesPor100mil")))

## 3b. Exportar a silver para o PostgreSQL (caminho A)
O DW continua no PostgreSQL. Esta célula gera o arquivo `silver_populacao_municipio.csv` no Volume; baixe-o (Catalog → Volume) e salve na pasta `sql/`, ao lado do `LAKE_IBGE_FIR.sql`.

In [ ]:
def exportar_silver(nome="silver_populacao_municipio.csv"):
    pdf = spark.table(T("silver_populacao_municipio")).orderBy("CodigoMunicipioIBGE").toPandas()
    caminho = f"{VOLUME_PATH}/{nome}"
    pdf.to_csv(caminho, index=False, encoding="utf-8")   # colunas = tabela lake_ibge.curated_populacao_municipio
    print("exportado:", caminho, "-", len(pdf), "linhas")

exportar_silver()

### Verificações (equivalem às consultas de validação do SQL)

In [ ]:
n_end = spark.table(T("dim_endereco")).count()
n_sem = spark.table(T("dim_endereco")).filter("PopulacaoMunicipio IS NULL").count()
print(f"endereços: {n_end} | sem população: {n_sem}")
assert n_end == spark.table(T("bronze_oper_endereco")).count(), "LEFT JOIN não pode duplicar nem perder endereços"
assert spark.table(T("silver_populacao_municipio")).groupBy("CodigoMunicipioIBGE").count().filter("count > 1").count() == 0
print("ok")

In [ ]:
import matplotlib.pyplot as plt
pdf = spark.table(T("gold_penetracao_municipio")).orderBy("SugestoesPor100mil").toPandas()
plt.figure(figsize=(7, 3.5))
plt.barh(pdf["Municipio"], pdf["SugestoesPor100mil"], color="#1F5FA6")
plt.xlabel("Sugestões de rota por 100 mil habitantes")
plt.title("Demanda reprimida por município (dados ilustrativos)")
plt.tight_layout(); plt.show()

## 4. Atualização incremental (SCD Tipo 1)
No SQL: *append* no raw → `REFRESH MATERIALIZED VIEW` → `UPDATE dw_fir.Endereco ... IS DISTINCT FROM`.
Aqui: nova versão do CSV vira um novo arquivo no Volume → *append* na bronze → recalcula a silver (que sempre fica com a linha mais recente) → atualiza a dimensão.

In [ ]:
# Simula uma "republicação" do IBGE: Nova Iguaçu com população diferente e Petrópolis nova
CSV_V2 = """codigo_municipio,municipio,uf,populacao,ano_referencia,fonte
3303500,Nova Iguaçu,RJ,790000,2025,Simulacao de republicacao (valor ilustrativo)
3303906,Petrópolis,RJ,278881,2022,IBGE - Censo Demografico 2022
"""
caminho_v2 = gravar("ibge_populacao_municipios_v2.csv", CSV_V2)
ingerir_bronze_ibge("ibge_populacao_municipios_v2.csv")
construir_silver_populacao()
display(spark.table(T("bronze_ibge_populacao")).select("codigo_municipio","municipio","populacao","ano_referencia","_arquivo"))
exportar_silver("silver_populacao_municipio_v2.csv")   # para o ETL incremental em SQL

In [ ]:
antes = spark.table(T("dim_endereco")).filter("EnderecoMunicipio = 'Nova Iguacu'").select("PopulacaoMunicipio").first()[0]
construir_dim_endereco()          # Tipo 1: sobrescreve com o valor atual
depois = spark.table(T("dim_endereco")).filter("EnderecoMunicipio = 'Nova Iguacu'").select("PopulacaoMunicipio").first()[0]
print("Nova Iguaçu:", antes, "->", depois)
assert antes != depois

### Alternativa em Delta: `MERGE INTO`
Em tabelas Delta (padrão no Databricks) o equivalente direto do `UPDATE ... IS DISTINCT FROM` do SQL é o `MERGE`. Útil quando a dimensão é grande e não queremos reescrevê-la inteira.
(Esta célula só roda no Databricks/Delta.)

In [ ]:
spark.sql(f"""
MERGE INTO {T("dim_endereco")} d
USING {T("silver_populacao_municipio")} p
  ON {CATALOG}.{SCHEMA}.dim_endereco.CodigoIBGEMunicipio = p.CodigoMunicipioIBGE
WHEN MATCHED AND NOT (d.PopulacaoMunicipio <=> p.PopulacaoResidente) THEN UPDATE SET
  d.PopulacaoMunicipio = p.PopulacaoResidente,
  d.AnoPopulacaoMunicipio = p.AnoReferencia
""")